In [261]:
import numpy as np
import cv2
import plotly.graph_objects as go
from scipy.linalg import circulant

In [262]:
# ── Paramètres ────────────────────────────────────────────────────────────────
IMG_PATH   = "im12.png"   # <-- remplace par ton image
K          = 200           # nombre de points du snake
ALPHA      = 2            # rigidité (élasticité)
BETA       = 3          # rigidité (courbure)
GAMMA      = 8           # force externe (gradient)
DT         = 0.1          # pas de temps
N_ITER     = 5000         # iterations totales
N_FRAMES   = 100           # nombre de frames dans l'animation
FRAME_DUR  = 60           # durée d'une frame en ms

In [263]:
# ── Image ─────────────────────────────────────────────────────────────────────
img = cv2.imread(IMG_PATH)
if img is None:
    raise FileNotFoundError(f"Image introuvable : {IMG_PATH}")

img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img_gray = cv2.normalize(img_gray, None, 0, 1.0,
                          cv2.NORM_MINMAX, dtype=cv2.CV_32F)
 
H, W = img_gray.shape
 

In [264]:
# ── Matrices internes ─────────────────────────────────────────────────────────
def make_A(K, alpha, beta, dt):
    row = np.zeros(K)
    row[0] = -2; row[1] = 1; row[-1] = 1
    D2 = circulant(row)
 
    row = np.zeros(K)
    row[0] = 6; row[1] = -4; row[-1] = -4; row[2] = 1; row[-2] = 1
    D4 = circulant(row)
 
    D = alpha * D2 - beta * D4
    return np.linalg.inv(np.eye(K) - dt * D)
 
A = make_A(K, ALPHA, BETA, DT)

In [265]:
# ── Gradient de l'image ───────────────────────────────────────────────────────
gradIy, gradIx = np.gradient(img_gray)
gradI          = np.sqrt(gradIx**2 + gradIy**2)
Gy, Gx         = np.gradient(gradI)
 

In [266]:
# ── Initialisation circulaire ─────────────────────────────────────────────────
cx = W / 2
cy = H / 2
R  = min(H, W) / 2.5
 
theta = np.linspace(0, 2 * np.pi, K, endpoint=False)
x = cx + R * np.cos(theta)
y = cy + R * np.sin(theta)

In [267]:
# ── Simulation – on sauvegarde N_FRAMES snapshots ─────────────────────────────
steps_per_frame = max(1, N_ITER // N_FRAMES)
 
snapshots_x = [x.copy()]
snapshots_y = [y.copy()]
 
for it in range(N_ITER):
    xi = np.clip(x.astype(int), 0, W - 1)
    yi = np.clip(y.astype(int), 0, H - 1)
 
    x = A @ (x + DT * GAMMA * Gx[yi, xi])
    y = A @ (y + DT * GAMMA * Gy[yi, xi])
 
    if (it + 1) % steps_per_frame == 0:
        snapshots_x.append(x.copy())
        snapshots_y.append(y.copy())
 
print(f"Simulation terminée : {len(snapshots_x)} frames capturées.")

Simulation terminée : 101 frames capturées.


In [268]:
# ── Construction de l'animation Plotly ───────────────────────────────────────
# Image de fond encodée en base64 pour Plotly
import base64, io
from PIL import Image as PILImage
 
pil_img  = PILImage.fromarray(img)
buf      = io.BytesIO()
pil_img.save(buf, format="PNG")
b64_img  = base64.b64encode(buf.getvalue()).decode()
 
SNAKE_COLOR = "#00e5ff"   # couleur fixe du snake
 
# ── Frame initiale ────────────────────────────────────────────────────────────
init_xs = list(snapshots_x[0]) + [snapshots_x[0][0]]
init_ys = list(snapshots_y[0]) + [snapshots_y[0][0]]
 
fig = go.Figure()
 
# Trace snake (sera mise à jour dans chaque frame)
fig.add_trace(go.Scatter(
    x=init_xs, y=init_ys,
    mode="lines",
    line=dict(color=SNAKE_COLOR, width=2),
    name="snake",
))
 
# ── Frames ────────────────────────────────────────────────────────────────────
frames = []
for i, (sx, sy) in enumerate(zip(snapshots_x, snapshots_y)):
    xs = list(sx) + [sx[0]]
    ys = list(sy) + [sy[0]]
    it_num = min(i * steps_per_frame, N_ITER)
    frames.append(go.Frame(
        data=[go.Scatter(
            x=xs, y=ys,
            mode="lines",
            line=dict(color=SNAKE_COLOR, width=2),
        )],
        name=str(i),
        layout=go.Layout(title_text=f"Itération {it_num} / {N_ITER}"),
    ))
 
fig.frames = frames
 
 

In [269]:
# ── Mise en page ───────────────────────────────────────────────────────────────
fig.update_layout(
    title=f"Snake actif — itération 0 / {N_ITER}",
    width=W * 2 + 80,
    height=H * 2 + 160,
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor="#111",
    plot_bgcolor="#111",
    font=dict(color="white"),
    xaxis=dict(
        range=[0, W], showgrid=False, zeroline=False,
        showticklabels=False, scaleanchor="y",
    ),
    yaxis=dict(
        range=[H, 0], showgrid=False, zeroline=False,
        showticklabels=False,
    ),
    images=[dict(
        source=f"data:image/png;base64,{b64_img}",
        xref="x", yref="y",
        x=0, y=0,
        sizex=W, sizey=H,
        sizing="stretch",
        layer="below",
        opacity=1.0,
    )],
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            x=0.2, xanchor="left",
            y=0, yanchor="top",
            pad=dict(t=0, r=0),
            buttons=[dict(
                label="▶  Play",
                method="animate",
                args=[None, dict(
                    frame=dict(duration=FRAME_DUR, redraw=True),
                    fromcurrent=True,
                    transition=dict(duration=0),
                )],
            )],
        ),
        dict(
            type="buttons",
            showactive=False,
            x=0.8, xanchor="right",
            y=0, yanchor="top",
            pad=dict(t=0, r=0),
            buttons=[dict(
                label="⏸  Pause",
                method="animate",
                args=[[None], dict(
                    frame=dict(duration=0, redraw=False),
                    mode="immediate",
                    transition=dict(duration=0),
                )],
            )],
        ),
    ],
    sliders=[dict(
        steps=[
            dict(
                method="animate",
                args=[[str(i)], dict(
                    mode="immediate",
                    frame=dict(duration=FRAME_DUR, redraw=True),
                    transition=dict(duration=0),
                )],
                label=str(min(i * steps_per_frame, N_ITER)),
            )
            for i in range(len(frames))
        ],
        active=0,
        x=0.05, y=0,
        len=0.9,
        currentvalue=dict(
            prefix="Itération : ",
            font=dict(size=13, color="white"),
            visible=True,
            xanchor="center",
        ),
        transition=dict(duration=0),
        pad=dict(b=10, t=45),
        bgcolor="#333",
        bordercolor="#555",
        tickcolor="white",
        font=dict(color="white"),
    )],
)
 
fig.show()